# 1. Exploratory analysis

This notebook rebuilds the core analysis: loading and validating the official NPS 2025 package, top and bottom reporting units, annual trends, designations, seasonal patterns, correlations, and DuckDB SQL. The study population is NPS units reporting visitor-use statistics, not all parks visited by the public.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.analysis import load_recreation_visits, load_park_metadata, annual_visits, add_metadata, top_bottom, duckdb_annual_summary, comparable_seasonality
raw = ROOT / 'data' / 'raw'
visits = load_recreation_visits(raw / 'Main_Data.csv')
metadata = load_park_metadata(raw / 'nps_parks_api.json')
annual = add_metadata(annual_visits(visits), metadata)
print(visits.shape, visits.UnitCode.nunique(), visits.Year.min(), visits.Year.max())

(193692, 5) 406 1979 2025


/var/folders/vp/wjhhgw095fj3cnz55r0q960c0000gn/T/ipykernel_66007/1561470319.py:9: UserWarning: Quarantining 4 negative TRV records; affected annual totals are partial
  visits = load_recreation_visits(raw / 'Main_Data.csv')


## Top and bottom 10
The question is which reporting units have the largest and smallest reported recreation-visit totals. Low values can reflect reporting practices as well as low use.

In [2]:
latest_year = int(annual.Year.max())
top, bottom = top_bottom(annual, latest_year)
display(top[['ParkName', 'Designation', 'RecreationVisits']])
display(bottom[['ParkName', 'Designation', 'RecreationVisits']])

,ParkName,Designation,RecreationVisits
1438,Blue Ridge,Parkway,16533753
6535,Golden Gate,National Recreation Area,15748676
6934,Great Smoky Mountains,National Park,11527939
10946,Natchez Trace,Parkway,7994783
9748,Lincoln Memorial,NaN,7743295
6112,Gateway,National Recreation Area,7696939
7075,Gulf Islands,National Seashore,7576923
7216,George Washington,Memorial Parkway,6732932
622,Appalachian,National Scenic Trail,6215118
9372,Lake Mead,National Recreation Area,6135586


,ParkName,Designation,RecreationVisits
4159,Edgar Allan Poe,National Historic Site,0
2560,Carter G. Woodson Home,National Historic Site,20
12744,Rio Grande,Wild & Scenic River,120
433,Aniakchak,National Monument & Preserve,247
208,Alagnak,Wild River,375
12338,Port Chicago Naval Magazine,National Memorial,782
7436,Harriet Tubman,National Historical Park,941
16098,Yukon - Charley Rivers,National Preserve,962
14312,Thaddeus Kosciuszko,National Memorial,1664
979,Bering Land Bridge,National Preserve,2674


## Annual trend and 2020
Annual totals show volume, not unique people or crowding. The 2020 break is descriptive and should not be assigned a causal explanation from this dataset alone.

In [3]:
annual_total = annual.groupby('Year', as_index=False).RecreationVisits.sum()
display(annual_total[annual_total.Year.isin([2019, 2020, 2021, 2024, 2025])])
ax = annual_total.plot(x='Year', y='RecreationVisits', figsize=(12, 5), marker='o', legend=False)
ax.set(title='NPS recreation visits over time', ylabel='Recreation visits')
plt.show()

,Year,RecreationVisits
40,2019,327516619
41,2020,237064332
42,2021,297115406
45,2024,331863358
46,2025,323014305


/var/folders/vp/wjhhgw095fj3cnz55r0q960c0000gn/T/ipykernel_66007/3882537965.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Seasonal pattern and DuckDB
Use 2023–2025 units with twelve valid months and positive totals in all three years. Average monthly proportions with equal weight per unit-year; compare with volume-weighted shares on the same cohort. This avoids mixing decades of changing coverage. Zero-total years have undefined proportions and are excluded only from this comparison.

In [4]:
monthly = comparable_seasonality(visits)
display(monthly)
ax = monthly.plot(x='Month', y=['MeanUnitShare', 'VolumeShare'], marker='o')
ax.set(title='2023–2025 comparable-unit seasonality', ylabel='Share of annual visits')
plt.show()
display(duckdb_annual_summary(annual).head())

,Month,MeanUnitShare,Volume,VolumeShare,Units
0,1,0.040124,39800200,0.040862,392
1,2,0.047318,44760171,0.045954,392
2,3,0.073106,71052425,0.072948,392
3,4,0.079443,80140256,0.082278,392
4,5,0.099415,96498398,0.099073,392
5,6,0.123851,117513212,0.120648,392
6,7,0.140768,125425505,0.128771,392
7,8,0.118833,113006615,0.116021,392
8,9,0.097812,95254767,0.097796,392
9,10,0.076290,84233206,0.086480,392


/var/folders/vp/wjhhgw095fj3cnz55r0q960c0000gn/T/ipykernel_66007/2156748888.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Year,RecreationVisits,ReportingUnits
0,1979,205369795.0,271
1,1980,220463211.0,275
2,1981,238592669.0,280
3,1982,244924579.0,290
4,1983,243619396.0,294


## Interpretation
The NPS series recovered substantially after 2020 and reached 323.0 million reported recreation visits in 2025. Rankings vary by year, and designation categories are not interchangeable: National Park units are one subset of the National Park System. Acreage, unique visitors, and direct crowding measures are not available in the verified inputs.